# Fixture Catalog and ranker

This notebook runs against the committed OSM fixture `tests/fixtures/catalog.osm`.
It does not download Geofabrik extracts and does not call a live Valhalla instance.

## What you are looking at

`dayhike` does not pull OSM when you rank. You ingest a snapshot **once** into a
SQLite file (the Catalog). Ranking reads that file. The fixture is a tiny
hand-built OSM XML oracle — two real DayHikes plus three relations that must be
dropped — not a Croatia dataset.

| Relation | Name | Fate |
| --- | --- | --- |
| 1001 | Ridge Walk | kept (~10 km, amenities, elevation, `mountain_hiking`) |
| 1002 | Valley Path | kept (~5 km, no amenities, no `sac_scale`, no `ele`) |
| 1003 | unnamed | dropped |
| 1004 | Long Traverse | dropped (too long) |
| 1005 | Via Dinarica Red | dropped (`network=iwn`) |

A production Catalog is the same pipeline on osmium-filtered Geofabrik XML.
See the README section *Pulling a real extract*.


In [ ]:
import tempfile
from pathlib import Path

from dayhike import build_catalog

repo = Path.cwd().resolve()
for candidate in (repo, *repo.parents):
    fixture = candidate / "tests" / "fixtures" / "catalog.osm"
    if fixture.is_file():
        repo = candidate
        break
else:
    raise FileNotFoundError("tests/fixtures/catalog.osm")

catalog_path = Path(tempfile.mkdtemp()) / "catalog.sqlite"
catalog = build_catalog(fixture, catalog_path)
print(catalog.ingest_report)
print("sqlite", catalog.path)
print("attribution", catalog.attribution)
for hike in catalog.dayhikes:
    print(hike.name)
    print(hike.feature_vector)


## The Catalog is SQLite

`build_catalog` wrote a database at `catalog.path`. There is no server.
`Catalog.open` reloads it; you can also query with `sqlite3`.


In [ ]:
import sqlite3

print("reopen", type(catalog).__name__, len(catalog.dayhikes), "DayHikes")
connection = sqlite3.connect(catalog.path)
connection.row_factory = sqlite3.Row
print("tables", sorted(row[0] for row in connection.execute(
    "SELECT name FROM sqlite_master WHERE type='table'"
)))
print("dayhikes:")
for row in connection.execute("SELECT id, name, length_m, difficulty FROM dayhikes"):
    print(dict(row))
print("Ridge Walk amenities:")
for row in connection.execute(
    "SELECT kind, count, nearest_m FROM amenity_facts WHERE dayhike_id = 1001"
):
    print(dict(row))
print("Valley Path amenities (zeros are stored):")
for row in connection.execute(
    "SELECT kind, count, nearest_m FROM amenity_facts WHERE dayhike_id = 1002"
):
    print(dict(row))
connection.close()


## DayHike objects

Each kept relation becomes a `DayHike`: Trailhead lat/lon, length, Difficulty
(max member-way `sac_scale`, or missing — never silently T1), AmenityFacts
within 20 km of the Trailhead, and an intrinsic FeatureVector. DriveTime is not
on that vector.


In [ ]:
from dayhike import resolve_seed

for hike in catalog.dayhikes:
    facts = hike.trailhead.amenity_facts
    print(
        hike.id,
        hike.name,
        f"trailhead=({hike.trailhead.latitude}, {hike.trailhead.longitude})",
        f"length_m={hike.length_m:.1f}",
        f"sac_scale={hike.difficulty.sac_scale!r}",
        f"huts={facts.hut.count}",
        f"parking={facts.parking.count}",
        f"cafe={facts.restaurant_cafe.count}",
        f"water={facts.drinking_water.count}",
    )

print("seed id", resolve_seed(catalog, "  RIDGE WALK  "))
print("feature names", catalog.scaler.feature_names)


## Ranking by Seed

One Seed returns the other in-band DayHike. Two dissimilar Seeds score each
candidate by **maximum** similarity to any Seed, not the mean — otherwise a
ridge and a valley would collapse into a mushy middle. The fixture only has
two DayHikes, so the max-not-mean overlay adds two extra candidates on top of
that Catalog.


In [ ]:
from dataclasses import replace

from dayhike import rank_dayhikes

for item in rank_dayhikes(catalog, ["Ridge Walk"]).items:
    print(f"one_seed\t{item.dayhike.name}\t{item.score}")

for item in rank_dayhikes(catalog, ["Valley Path"]).items:
    print(f"one_seed_valley\t{item.dayhike.name}\t{item.score}")

ridge, valley = catalog.dayhikes
near_ridge = replace(ridge, id=9001, name="Near Ridge")
midpoint = replace(
    valley,
    id=9002,
    name="Mid Path",
    length_m=(ridge.length_m + valley.length_m) / 2,
    feature_vector=replace(
        valley.feature_vector,
        length_m=(ridge.feature_vector.length_m + valley.feature_vector.length_m) / 2,
        elevation_min_m=ridge.feature_vector.elevation_min_m,
        elevation_max_m=ridge.feature_vector.elevation_max_m,
        elevation_gain_m=ridge.feature_vector.elevation_gain_m,
        difficulty_ordinal=1,
    ),
)
extended = replace(catalog, dayhikes=(*catalog.dayhikes, near_ridge, midpoint))
only_ridge = {
    item.dayhike.name: item.score
    for item in rank_dayhikes(extended, ["Ridge Walk"]).items
}
only_valley = {
    item.dayhike.name: item.score
    for item in rank_dayhikes(extended, ["Valley Path"]).items
}
both = rank_dayhikes(extended, ["Ridge Walk", "Valley Path"])
for item in both.items:
    print(f"two_seeds\t{item.dayhike.name}\t{item.score}")
for name in ("Near Ridge", "Mid Path"):
    score = next(item.score for item in both.items if item.dayhike.name == name)
    expected_max = max(only_ridge[name], only_valley[name])
    expected_mean = (only_ridge[name] + only_valley[name]) / 2
    print(f"max_not_mean\t{name}\t{score}\t{expected_max}\t{expected_mean}")


## Unknown Seed

A Seed name that is not in the Catalog fails loudly. There is no fuzzy lookup.


In [ ]:
from dayhike import UnknownSeedName, rank_dayhikes

try:
    rank_dayhikes(catalog, ["Unknown Ridge"])
except UnknownSeedName as exc:
    print(type(exc).__name__)
    print(exc)


## DriveTime filter (stub client)

DriveTime is a query-time filter from today's Origin, not a FeatureVector column. This cell stubs the Valhalla matrix client so CI never needs routing tiles. Real routing from a live Origin needs local Valhalla tiles.

Valhalla is **not** a pip package and is **not** started by this notebook. It is a Docker service (`data/valhalla/compose.yaml`) that listens on `http://127.0.0.1:8002` after `prepare-roads.sh` writes a car-only PBF into `data/valhalla/` and you run compose. `dayhike` then POSTs `/sources_to_targets`. Ranking without Origin never connects.


In [ ]:
from dayhike import Origin, filter_by_drive_time, rank_dayhikes

class StubMatrix:
    def __init__(self, times_s):
        self._times_s = times_s

    def sources_to_targets(self, origin, trailheads):
        return self._times_s

origin = Origin(latitude=45.8, longitude=15.97)
ranking = rank_dayhikes(catalog, ["Ridge Walk"])
kept = filter_by_drive_time(ranking, origin, 2, StubMatrix((1800.0,)))
for item in kept.items:
    print(f"drive_keep\t{item.dayhike.name}\t{item.drive_time.hours}")
dropped = filter_by_drive_time(ranking, origin, 2, StubMatrix((None,)))
print(f"drive_drop_count\t{len(dropped.items)}")


## Limitations

- The Catalog unit is named `route=hiking` relations in a length band (2–25 km). Unnamed paths never become DayHikes; neither do superroutes or out-of-band lengths.
- DriveTime for a real Origin needs local Valhalla tiles; the stub above is only a stand-in.
- There are no scraped reviews and no webpage ratings yet. Ratings and UI are future work.
- Ingest reads OSM XML or PBF (pyosmium). `dayhike fetch-extracts` downloads the Geofabrik list.
- AmenityFacts count nodes, ways, and tagged relations within 20 km of the Trailhead.
